# 5.2 Speech Trial 002: PyCaret Benchmark

Purpose:
- Compare multiple classical ML models on the speech feature table.
- Use the existing speaker-safe split.
- Evaluate the selected PyCaret model on the held-out test set.

Important:
PyCaret is used for benchmarking classical ML models. The final test evaluation remains our own speaker-held-out evaluation.

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from pycaret.classification import (
    setup,
    compare_models,
    pull,
    predict_model,
    save_model,
)

from rural_stroke_assist.config import (
    SPEECH_FEATURES_PATH,
    SPEECH_TRIAL_002_MODEL_PATH,
    SPEECH_TRIAL_002_RESULTS_PATH,
    SPEECH_TRIAL_002_LEADERBOARD_PATH,
    SPEECH_TRIAL_002_PREDICTIONS_PATH,
    SPEECH_TRIAL_002_CONFUSION_MATRIX_PATH,
)

from rural_stroke_assist.modeling.pycaret_speech import (
    split_pycaret_speech_data,
    build_pycaret_train_data,
    prepare_pycaret_speech_dataframe,
)

from rural_stroke_assist.features.speech_features import (
    SPEECH_INVERSE_LABEL_MAPPING,
)

from rural_stroke_assist.modeling.evaluation import (
    evaluate_binary_classifier,
    create_classification_report_dict,
    save_confusion_matrix_plot,
    save_markdown_results,
)

In [3]:
speech_features_df = pd.read_csv(SPEECH_FEATURES_PATH)

speech_features_df.head()

,path,file_name,label,label_encoded,speaker_id,torgo_group,split,mfcc_1_mean,mfcc_2_mean,mfcc_3_mean,...,rms_mean,rms_std,zcr_mean,zcr_std,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std
0,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0001.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-163.494843,56.581371,-30.107115,...,0.144104,0.102247,0.151883,0.121665,2042.334292,781.147275,1723.756351,442.497088,3784.583997,1448.862083
1,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0002.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-196.675262,66.341858,-25.319544,...,0.108665,0.071716,0.154005,0.139137,1910.950448,865.713928,1632.424714,357.816673,3471.785430,1342.273322
2,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0003.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-181.038422,52.951725,-29.189020,...,0.132562,0.098360,0.232347,0.199550,2342.658893,1164.214791,1703.955806,427.230919,3979.150080,1727.665199
3,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0004.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-376.797882,17.045906,36.214321,...,0.007747,0.007332,0.417319,0.225873,3702.937939,1098.628999,2367.278508,269.317380,6438.893312,978.800780
4,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,wav_arrayMic_M01S01_0005.wav,dysarthric,1,wav_arrayMic_M01S01,M_Dys,train,-226.536652,40.652927,-18.713095,...,0.097521,0.098568,0.229228,0.181667,2410.353555,1014.842146,1835.126339,420.014444,4338.126990,1615.570376


In [4]:
speech_features_df.groupby(["split", "label"]).size()

split  label     
test   control       1867
       dysarthric     982
train  control       6678
       dysarthric    4221
val    control       2909
       dysarthric     973
dtype: int64

In [5]:
train_df, val_df, test_df = split_pycaret_speech_data(speech_features_df)

pycaret_train_source_df = build_pycaret_train_data(train_df, val_df)

len(train_df), len(val_df), len(test_df), len(pycaret_train_source_df)

(10899, 3882, 2849, 14781)

In [6]:
pycaret_train_df, feature_columns = prepare_pycaret_speech_dataframe(
    pycaret_train_source_df
)

pycaret_test_df, _ = prepare_pycaret_speech_dataframe(
    test_df
)

pycaret_train_df.head(), len(feature_columns)

(   mfcc_1_mean  mfcc_2_mean  mfcc_3_mean  mfcc_4_mean  mfcc_5_mean  \
 0  -163.494843    56.581371   -30.107115    19.472561    11.054909   
 1  -196.675262    66.341858   -25.319544    28.143564    14.642306   
 2  -181.038422    52.951725   -29.189020    19.459269    13.609412   
 3  -376.797882    17.045906    36.214321     3.459953     9.538445   
 4  -226.536652    40.652927   -18.713095    30.717720    24.219467   
 
    mfcc_6_mean  mfcc_7_mean  mfcc_8_mean  mfcc_9_mean  mfcc_10_mean  ...  \
 0    -2.725810    -4.639313   -13.518082   -17.710405     -0.082665  ...   
 1   -15.741869    -5.040081   -19.771603   -29.083685     -9.343624  ...   
 2    -3.706669   -13.559193   -17.712330   -15.015168     -2.159306  ...   
 3    -0.999489    -1.124271     6.001362     4.602589      4.865216  ...   
 4    -9.053682    -6.546130    -9.679409   -15.085834     -7.274210  ...   
 
     rms_std  zcr_mean   zcr_std  spectral_centroid_mean  \
 0  0.102247  0.151883  0.121665             204

In [7]:
experiment = setup(
    data=pycaret_train_df,
    target="label_encoded",
    session_id=42,
    normalize=True,
    fix_imbalance=False,
    fold=5,
    html=False,
    verbose=True,
)

                    Description             Value
0                    Session id                42
1                        Target     label_encoded
2                   Target type            Binary
3           Original data shape       (14781, 38)
4        Transformed data shape       (14781, 38)
5   Transformed train set shape       (10346, 38)
6    Transformed test set shape        (4435, 38)
7              Numeric features                37
8                    Preprocess              True
9               Imputation type            simple
10           Numeric imputation              mean
11       Categorical imputation              mode
12                    Normalize              True
13             Normalize method            zscore
14               Fold Generator   StratifiedKFold
15                  Fold Number                 5
16                     CPU Jobs                -1
17                      Use GPU             False
18               Log Experiment             False


In [8]:
best_models = compare_models(
    sort="F1",
    n_select=5,
)

                                    Model  Accuracy     AUC  Recall   Prec.  \
lightgbm  Light Gradient Boosting Machine    0.9445  0.9884  0.8812  0.9576   
knn                K Neighbors Classifier    0.9452  0.9813  0.8603  0.9815   
et                 Extra Trees Classifier    0.9397  0.9892  0.8413  0.9849   
rf               Random Forest Classifier    0.9329  0.9840  0.8377  0.9670   
gbc          Gradient Boosting Classifier    0.9035  0.9601  0.7750  0.9401   
dt               Decision Tree Classifier    0.8698  0.8558  0.8089  0.8187   
qda       Quadratic Discriminant Analysis    0.8710  0.9402  0.7517  0.8635   
ada                  Ada Boost Classifier    0.8467  0.9009  0.7145  0.8259   
lr                    Logistic Regression    0.8244  0.8780  0.6502  0.8129   
lda          Linear Discriminant Analysis    0.8222  0.8767  0.6268  0.8252   
svm                   SVM - Linear Kernel    0.8168  0.8669  0.6433  0.7979   
ridge                    Ridge Classifier    0.8206 

In [9]:
leaderboard_df = pull()

SPEECH_TRIAL_002_LEADERBOARD_PATH.parent.mkdir(parents=True, exist_ok=True)
leaderboard_df.to_csv(SPEECH_TRIAL_002_LEADERBOARD_PATH, index=False)

leaderboard_df

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.9445,0.9884,0.8812,0.9576,0.9178,0.8760,0.8778,0.176
knn,K Neighbors Classifier,0.9452,0.9813,0.8603,0.9815,0.9169,0.8763,0.8805,0.988
et,Extra Trees Classifier,0.9397,0.9892,0.8413,0.9849,0.9074,0.8631,0.8690,0.122
rf,Random Forest Classifier,0.9329,0.9840,0.8377,0.9670,0.8977,0.8482,0.8531,0.502
gbc,Gradient Boosting Classifier,0.9035,0.9601,0.7750,0.9401,0.8495,0.7795,0.7875,2.242
dt,Decision Tree Classifier,0.8698,0.8558,0.8089,0.8187,0.8136,0.7136,0.7138,0.432
qda,Quadratic Discriminant Analysis,0.8710,0.9402,0.7517,0.8635,0.8036,0.7082,0.7121,0.026
ada,Ada Boost Classifier,0.8467,0.9009,0.7145,0.8259,0.7661,0.6530,0.6568,0.458
lr,Logistic Regression,0.8244,0.8780,0.6502,0.8129,0.7222,0.5962,0.6043,0.890
lda,Linear Discriminant Analysis,0.8222,0.8767,0.6268,0.8252,0.7123,0.5871,0.5988,0.022


In [10]:
best_model = best_models[0] if isinstance(best_models, list) else best_models

best_model

LGBMClassifier(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
               importance_type='split', learning_rate=0.1, max_depth=-1,
               min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
               n_estimators=100, n_jobs=-1, num_leaves=31, objective=None,
               random_state=42, reg_alpha=0.0, reg_lambda=0.0, subsample=1.0,
               subsample_for_bin=200000, subsample_freq=0)

In [11]:
pycaret_predictions = predict_model(
    best_model,
    data=pycaret_test_df,
)

pycaret_predictions.head()

                             Model  Accuracy    AUC  Recall   Prec.      F1  \
0  Light Gradient Boosting Machine    0.7848  0.927  0.9348  0.6258  0.7497   

    Kappa     MCC  
0  0.5736  0.6093  


,mfcc_1_mean,mfcc_2_mean,mfcc_3_mean,mfcc_4_mean,mfcc_5_mean,mfcc_6_mean,mfcc_7_mean,mfcc_8_mean,mfcc_9_mean,mfcc_10_mean,...,zcr_std,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std,label_encoded,prediction_label,prediction_score
1850,-372.027496,83.220184,6.564825,24.824373,6.393429,2.086060,-5.711854,0.200638,-8.851834,3.535347,...,0.052434,1836.931152,929.404114,1782.802856,545.772644,3691.731689,1979.846436,1,1,0.9746
1851,-331.828918,93.055191,-19.952835,21.880646,13.370432,-2.531636,-8.846474,-4.595466,-0.589336,4.524516,...,0.043686,1690.620728,740.930969,1629.959961,496.470764,3391.447266,1586.069946,1,1,0.9708
1852,-309.253204,80.349823,-15.267198,31.479836,17.605717,-6.662419,-8.877670,-4.294051,-1.784879,-0.318184,...,0.071410,1684.828369,678.470947,1644.154785,471.921356,3516.649658,1489.170288,1,1,0.9965
1853,-350.950500,75.520081,2.589722,44.287037,18.498871,-3.871168,-8.363561,6.292027,-15.018199,-1.841695,...,0.047116,1645.737549,802.743469,1761.179443,481.487000,3557.617188,1648.860596,1,1,0.9937
1854,-348.846771,77.841995,-2.706631,19.650021,9.325848,7.125217,-2.586755,-2.748945,-13.228964,2.118490,...,0.113177,1863.905518,984.731567,1714.538818,541.105774,3682.235107,1903.017578,1,1,0.9593


In [12]:
pycaret_predictions.columns.tolist()

['mfcc_1_mean',
 'mfcc_2_mean',
 'mfcc_3_mean',
 'mfcc_4_mean',
 'mfcc_5_mean',
 'mfcc_6_mean',
 'mfcc_7_mean',
 'mfcc_8_mean',
 'mfcc_9_mean',
 'mfcc_10_mean',
 'mfcc_11_mean',
 'mfcc_12_mean',
 'mfcc_13_mean',
 'mfcc_1_std',
 'mfcc_2_std',
 'mfcc_3_std',
 'mfcc_4_std',
 'mfcc_5_std',
 'mfcc_6_std',
 'mfcc_7_std',
 'mfcc_8_std',
 'mfcc_9_std',
 'mfcc_10_std',
 'mfcc_11_std',
 'mfcc_12_std',
 'mfcc_13_std',
 'duration_seconds',
 'rms_mean',
 'rms_std',
 'zcr_mean',
 'zcr_std',
 'spectral_centroid_mean',
 'spectral_centroid_std',
 'spectral_bandwidth_mean',
 'spectral_bandwidth_std',
 'spectral_rolloff_mean',
 'spectral_rolloff_std',
 'label_encoded',
 'prediction_label',
 'prediction_score']

In [13]:
if "prediction_label" in pycaret_predictions.columns:
    pred_col = "prediction_label"
elif "Label" in pycaret_predictions.columns:
    pred_col = "Label"
else:
    raise ValueError("Could not find PyCaret prediction label column.")

if "prediction_score" in pycaret_predictions.columns:
    score_col = "prediction_score"
elif "Score" in pycaret_predictions.columns:
    score_col = "Score"
else:
    score_col = None

y_test = pycaret_test_df["label_encoded"].to_numpy()
y_pred = pycaret_predictions[pred_col].astype(int).to_numpy()

if score_col is not None:
    raw_score = pycaret_predictions[score_col].to_numpy()

    # PyCaret score can be confidence of predicted class, not always class-1 probability.
    # For safer ROC-AUC, use score only if predictions are class probabilities for positive class.
    y_score = raw_score
else:
    y_score = None

np.unique(y_pred, return_counts=True), np.unique(y_test, return_counts=True)

((array([0, 1]), array([1382, 1467], dtype=int64)),
 (array([0, 1], dtype=int64), array([1867,  982], dtype=int64)))

In [14]:
if hasattr(best_model, "predict_proba"):
    y_score = best_model.predict_proba(pycaret_test_df[feature_columns])[:, 1]
else:
    y_score = None

y_score[:5] if y_score is not None else None

array([0.48019045, 0.67102603, 0.63851122, 0.65322282, 0.72938228])

In [15]:
test_metrics = evaluate_binary_classifier(
    y_true=y_test,
    y_pred=y_pred,
    y_score=y_score,
)

test_metrics

{'accuracy': 0.7848367848367849,
 'precision': 0.6257668711656442,
 'recall': 0.9348268839103869,
 'f1': 0.749693752552062,
 'roc_auc': 0.5504962381244839}

In [16]:
target_names = [
    SPEECH_INVERSE_LABEL_MAPPING[0],
    SPEECH_INVERSE_LABEL_MAPPING[1],
]

report = create_classification_report_dict(
    y_true=y_test,
    y_pred=y_pred,
    target_names=target_names,
)

report

{'control': {'precision': 0.9536903039073806,
  'recall': 0.7059453668987681,
  'f1-score': 0.8113265620190828,
  'support': 1867.0},
 'dysarthric': {'precision': 0.6257668711656442,
  'recall': 0.9348268839103869,
  'f1-score': 0.749693752552062,
  'support': 982.0},
 'accuracy': 0.7848367848367849,
 'macro avg': {'precision': 0.7897285875365124,
  'recall': 0.8203861254045774,
  'f1-score': 0.7805101572855724,
  'support': 2849.0},
 'weighted avg': {'precision': 0.8406608862336759,
  'recall': 0.7848367848367849,
  'f1-score': 0.7900828207426299,
  'support': 2849.0}}

In [17]:
save_confusion_matrix_plot(
    y_true=y_test,
    y_pred=y_pred,
    class_names=target_names,
    output_path=SPEECH_TRIAL_002_CONFUSION_MATRIX_PATH,
)

SPEECH_TRIAL_002_CONFUSION_MATRIX_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/figures/experiments/speech_trial_002_pycaret_confusion_matrix.png')

In [18]:
SPEECH_TRIAL_002_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

predictions_df = test_df.reset_index(drop=True).copy()
predictions_df["predicted_label_encoded"] = y_pred
predictions_df["predicted_label"] = predictions_df["predicted_label_encoded"].map(
    SPEECH_INVERSE_LABEL_MAPPING
)
predictions_df["true_label"] = predictions_df["label_encoded"].map(
    SPEECH_INVERSE_LABEL_MAPPING
)
predictions_df["is_correct"] = (
    predictions_df["label_encoded"] == predictions_df["predicted_label_encoded"]
)

if y_score is not None:
    predictions_df["dysarthric_probability"] = y_score

predictions_df.to_csv(SPEECH_TRIAL_002_PREDICTIONS_PATH, index=False)

SPEECH_TRIAL_002_PREDICTIONS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/speech_trial_002_pycaret_predictions.csv')

In [19]:
save_markdown_results(
    metrics=test_metrics,
    report=report,
    output_path=SPEECH_TRIAL_002_RESULTS_PATH,
    title="Speech Trial 002 PyCaret Benchmark Results",
)

SPEECH_TRIAL_002_RESULTS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/experiments/speech_trial_002_pycaret_benchmark_results.md')

In [20]:
dysarthric_recall = report["dysarthric"]["recall"]
dysarthric_f1 = report["dysarthric"]["f1-score"]
roc_auc = test_metrics.get("roc_auc", 0)

accepted = (
    dysarthric_recall >= 0.75
    and dysarthric_f1 >= 0.70
    and roc_auc >= 0.80
)

{
    "dysarthric_recall": dysarthric_recall,
    "dysarthric_f1": dysarthric_f1,
    "roc_auc": roc_auc,
    "accepted": accepted,
}

{'dysarthric_recall': 0.9348268839103869,
 'dysarthric_f1': 0.749693752552062,
 'roc_auc': 0.5504962381244839,
 'accepted': False}

In [21]:
trial_001_dysarthric_f1 = 0.7705
trial_001_roc_auc = 0.9426

better_than_trial_001 = (
    dysarthric_f1 > trial_001_dysarthric_f1
    or roc_auc > trial_001_roc_auc
)

if accepted and better_than_trial_001:
    SPEECH_TRIAL_002_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_model(best_model, str(SPEECH_TRIAL_002_MODEL_PATH))
    print("Accepted improved PyCaret model saved to:", SPEECH_TRIAL_002_MODEL_PATH)
elif accepted:
    print("PyCaret model accepted but not better than Trial 001. Results saved; model not promoted.")
else:
    print("PyCaret model rejected. Results saved; model not saved.")

PyCaret model rejected. Results saved; model not saved.
